### 1. Import libraries and define constants and variables

In [64]:
import math
import random

# Constants
Q_LIMIT = 100
BUSY = 1
IDLE = 0

# Global variables
next_event_type = 0
num_custs_delayed = 0
num_delays_required = 0
num_events = 2
num_in_q = 0
server_status = IDLE

area_num_in_q = 0.0
area_server_status = 0.0
mean_interarrival = 0.0
mean_service = 0.0
sim_time = 0.0
time_last_event = 0.0

time_arrival = [0.0] * (Q_LIMIT + 1)
time_next_event = [0.0] * 3

total_of_delays = 0.0

### 2. Random exponential generator

In [65]:
def expon(mean):
    return -mean * math.log(random.random())

### 3. Initialize

In [66]:
def initialize():
    global sim_time, server_status, num_in_q, time_last_event
    global num_custs_delayed, total_of_delays
    global area_num_in_q, area_server_status

    sim_time = 0.0
    server_status = IDLE
    num_in_q = 0
    time_last_event = 0.0

    num_custs_delayed = 0
    total_of_delays = 0.0

    area_num_in_q = 0.0
    area_server_status = 0.0

    # Schedule first arrival
    time_next_event[1] = sim_time + expon(mean_interarrival)
    time_next_event[2] = float('inf')

### 4. Timing function

In [67]:
def timing():
    global sim_time, next_event_type

    min_time_next_event = float('inf')
    next_event_type = 0

    for i in range(1, num_events + 1):
        if time_next_event[i] < min_time_next_event:
            min_time_next_event = time_next_event[i]
            next_event_type = i

    if next_event_type == 0:
        raise Exception("Event list empty")

    sim_time = min_time_next_event

### 5. Arrival event

In [68]:
def arrive():
    global num_in_q, server_status, total_of_delays
    global num_custs_delayed

    # Schedule next arrival
    time_next_event[1] = sim_time + expon(mean_interarrival)

    if server_status == BUSY:
        num_in_q += 1

        if num_in_q > Q_LIMIT:
            raise Exception("Queue overflow")

        time_arrival[num_in_q] = sim_time

    else:
        delay = 0.0
        total_of_delays += delay

        num_custs_delayed += 1
        server_status = BUSY

        # Schedule departure
        time_next_event[2] = sim_time + expon(mean_service)


### 6. Departure event

In [69]:
def depart():
    global num_in_q, server_status, total_of_delays
    global num_custs_delayed

    if num_in_q == 0:
        server_status = IDLE
        time_next_event[2] = float('inf')

    else:
        num_in_q -= 1

        delay = sim_time - time_arrival[1]
        total_of_delays += delay

        num_custs_delayed += 1

        time_next_event[2] = sim_time + expon(mean_service)

        # Shift queue
        for i in range(1, num_in_q + 1):
            time_arrival[i] = time_arrival[i + 1]


### 7. Update stats

In [70]:
def update_time_avg_stats():
    global time_last_event, area_num_in_q, area_server_status

    time_since_last_event = sim_time - time_last_event
    time_last_event = sim_time

    area_num_in_q += num_in_q * time_since_last_event
    area_server_status += server_status * time_since_last_event


### 8. Report

In [71]:
def report():
    print("\nSingle-server queue system\n")
    print(f"Mean interarrival time: {mean_interarrival:.3f} minutes")
    print(f"Mean service time: {mean_service:.3f} minutes")
    print(f"Number of customers: {num_delays_required}\n")

    print(f"Average delay in queue: {total_of_delays / num_custs_delayed:.3f}")
    print(f"Average number in queue: {area_num_in_q / sim_time:.3f}")
    print(f"Server utilization: {area_server_status / sim_time:.3f}")
    print(f"Time simulation ended: {sim_time:.3f} minutes")

### 9. Main simulation

In [72]:
def main():
    global mean_interarrival, mean_service, num_delays_required

    # Input parameters (you can change these)
    mean_interarrival = 1.0
    mean_service = 0.5
    num_delays_required = 1000

    initialize()

    while num_custs_delayed < num_delays_required:
        timing()
        update_time_avg_stats()

        if next_event_type == 1:
            arrive()
        elif next_event_type == 2:
            depart()

    report()


if __name__ == "__main__":
    main()


Single-server queue system

Mean interarrival time: 1.000 minutes
Mean service time: 0.500 minutes
Number of customers: 1000

Average delay in queue: 0.513
Average number in queue: 0.525
Server utilization: 0.518
Time simulation ended: 976.753 minutes
